# 01 Data Audit

Load the original read-only workfiles and verify the sample windows and variables used by the main-paper notebooks.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import get_paths, ensure_output_dirs
paths = get_paths()
ensure_output_dirs(paths)


In [2]:
from src.data import load_cex_short, load_cex_cohort, audit_variables
from src.plots import FIG3_VARS
from src.metrics import write_table

df = load_cex_short(paths)
cohort = load_cex_cohort(paths)
print('all_CEX_data_short:', df.shape)
print('all_CEX_data_cohort:', cohort.shape)
print('short span:', df.period.min(), df.period.max())
print('cohort span:', cohort.period.min(), cohort.period.max())

all_CEX_data_short: (160, 79)
all_CEX_data_cohort: (159, 163)
short span: 1969Q1 2008Q4
cohort span: 1969Q2 2008Q4


In [3]:
key_vars = ['sh_rr', 'sh_pi', 'pi_trend_cg', 'PI_STAR_IRL_BACK'] + [v for v, _, _ in FIG3_VARS]
audit = audit_variables(df, key_vars)
audit

,variable,n,start,end,mean,std
0,sh_rr,160,1969Q1,2008Q4,-2.980232e-09,0.588040
1,sh_pi,159,1969Q2,2008Q4,3.598865e-04,0.285689
2,pi_trend_cg,160,1969Q1,2008Q4,4.048573e+00,1.901962
3,PI_STAR_IRL_BACK,142,1969Q1,2004Q2,4.011042e+00,1.779407
4,D_SD_LNYBTIMP2_SA,115,1980Q2,2008Q4,1.226237e-03,0.027837
5,D_SD_LNSALARYIMP_SA,115,1980Q2,2008Q4,4.423111e-04,0.034784
6,D_SD_LNTOTALEXP3_SA,116,1980Q1,2008Q4,4.967651e-05,0.011001
7,D_SD_LNCONS_SA,116,1980Q1,2008Q4,3.593138e-04,0.008720
8,D_GINI_YBTIMP2_SA,115,1980Q2,2008Q4,4.499102e-04,0.008775
9,D_GINI_SALARYIMP_SA,115,1980Q2,2008Q4,3.365986e-04,0.009300


In [4]:
write_table(audit, paths.tables / '01_variable_audit.csv')
missing = [c for c in key_vars if c not in df.columns]
assert not missing, missing
assert df['sh_rr'].notna().sum() == 160
assert df.loc[df['D_SD_LNCONS_SA'].notna(), 'period'].iloc[0].strftime('%YQ%q') == '1980Q1'
print('Audit passed; original workfiles were only read.')

Audit passed; original workfiles were only read.


## Deviation Note

The source CEX microdata files exist in the original folder, but this replication starts from `all_CEX_data_short.dta`. That is the authors' supplied final analysis dataset and already embeds the Stata/EViews seasonal-adjustment outputs used by the main-paper regressions.